<a href="https://colab.research.google.com/github/zhaw-iwi/LLM-Intro/blob/main/Ollama_on_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
! sudo apt-get install -y pciutils  #PCI-Dienstprogramme installieren, um angeschlossene Hardwarekomponenten wie Grafikkarten (GPUs) mit dem Befehl lspci zu identifizieren

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.10.0-2build1).
0 upgraded, 0 newly installed, 0 to remove and 43 not upgraded.


In [8]:
import subprocess                                               #Python-Standardbibliothek zur Ausführung von Terminalbefehlen
import requests                                                 #Externe Bibliothek zum Senden von HTTP-Anfragen

# Entferne das alte Installationsskript, falls es existiert
subprocess.run(['rm', '-f', 'install.sh'], check=False)         #Vorhandene Datei löschen, bei einem fehlerhaften Rückgabecode keine Exception auslösen

# Versuche, das Skript mit requests herunterzuladen
url = 'https://ollama.ai/install.sh'                            #URL des Ollama-Installationsskripts speichern
try:                                                            #Nachfolgenden Code ausführen und mögliche Fehler abfangen
    response = requests.get(url, stream=True)                   #HTTP-GET-Anfrage senden und Daten schrittweise herunterladen
    response.raise_for_status()                                 #Löst einen Fehler für schlechte Statuscodes aus (4xx oder 5xx)

    with open('install.sh', 'wb') as f:                         #Datei im binären Schreibmodus öffnen und das Dateiobjekt in f speichern
        for chunk in response.iter_content(chunk_size=8192):    #Heruntergeladene Daten in Blöcken von max. 8192 Bytes durchlaufen
            f.write(chunk)                                      #Jeweiligen Datenblock in die Datei schreiben

    print("Das Ollama-Installationsskript wurde erfolgreich heruntergeladen (mittels requests).")
except requests.exceptions.RequestException as e:               #Fehler bei der HTTP-Anfrage abfangen und in e speichern
    print(f"Fehler beim Herunterladen des Ollama-Installationsskripts mittels requests: {e}")
except Exception as e:                                          #Sonstige unerwartete Fehler abfangen und in e speichern
    print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")

Das Ollama-Installationsskript wurde erfolgreich heruntergeladen (mittels requests).


In [9]:
! sudo apt-get install -y zstd  #Zstandard installieren, um komprimierte Dateien bei der Ollama-Installation entpacken zu können

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 43 not upgraded.


In [10]:
! pip install ollama  #Python-Bibliothek ollama installieren, um lokale LLMs direkt aus Python anzusprechen

In [11]:
import os           #Zugriff auf Betriebssystemfunktionen und Umgebungsvariablen
import threading    #Parallele Ausführung von Aufgaben innerhalb eines Python-Prozesses
import subprocess   #Ausführung von Terminalbefehlen und Starten externer Programme
import requests     #Senden von HTTP-Anfragen an APIs und Herunterladen von Dateien
import json         #Umwandlung zwischen Python-Datenstrukturen und dem JSON-Format
import ollama       #Kommunikation mit lokal ausgeführten LLMs über die Ollama API

In [ ]:
# Überprüfe, ob das Skript heruntergeladen wurde und führe es aus
if os.path.exists('install.sh'):                                    #Prüfen, ob die Datei install.sh im aktuellen Arbeitsverzeichnis existiert
    print("Ausführen des Ollama-Installationsskripts...")
    # Stelle sicher, dass die Datei ausführbar ist
    subprocess.run(['chmod', '+x', 'install.sh'], check=True)       #Ausführungsberechtigung für das Skript erteilen
    # Führe das Skript aus
    try:
        result = subprocess.run(['./install.sh'], capture_output=True, text=True, check=True)   #Installationsskript ausführen und Ergebnis speichern
        print(result.stdout)                                                                    #Standardausgabe des Installationsskripts ausgeben
        print(result.stderr)                                                                    #Fehlerausgabe des Installationsskripts ausgeben
        print("Das Ollama-Installationsskript wurde erfolgreich ausgeführt.")
    except subprocess.CalledProcessError as e:                                                  #Fehler bei der Ausführung abfangen und Fehlerobjekt in e speichern
        print(f"Fehler beim Ausführen des Ollama-Installationsskripts: {e}")
        print(e.stdout)                                                                         #Standardausgabe des fehlgeschlagenen Prozesses ausgeben
        print(e.stderr)                                                                         #Fehlerausgabe des fehlgeschlagenen Prozesses ausgeben
else:                                                                                           #Wird ausgeführt, wenn die Datei install.sh nicht existiert
    print("Ollama-Installationsskript nicht gefunden. Bitte laden Sie es zuerst herunter.")

Ausführen des Ollama-Installationsskripts...

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                          

                                                                           0.6%
#                                                                          1.9%
###                                                                        4.4%
####                                                                       6.9%
######                                                                     9.4%
#########                                                                 12.5%
##########                                                                15.0%
############                                                              17.2%
#############                                                             19.0%
##############                                                            

In [13]:
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'     #Ollama API auf allen Netzwerkschnittstellen über Port 11434 erreichbar machen
os.environ['OLLAMA_ORIGINS'] = '*'              #Browser-Anfragen von beliebigen Ursprüngen erlauben
subprocess.Popen(["ollama", "serve"])           #Ollama-Server als separaten Prozess im Hintergrund starten

<Popen: returncode: None args: ['ollama', 'serve']>

Couldn't find '/home/codespace/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIKuWsd/Q6DChagUgEW2+H746nXSA9DUBB2i5XlHlP2Oq



time=2026-11-26T22:44:17.637Z level=INFO source=routes.go:2484 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/home/codespace/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:0 OLLAMA_ORIGINS:[* http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.

In [14]:
# @title Select your model
model = "deepseek-r1:7b" # @param ["deepseek-r1:1.5b","deepseek-r1:7b","deepseek-r1:14b","deepseek-r1:32b","deepseek-r1:70b","deepseek-coder:1.3b","deepseek-coder:6.7b","deepseek-coder:33b","gemma3:12b","gemma3:27b","llama3.3:70b","mistral:7b","phi4:14b","qwen2.5:7b","qwen2.5:14b","qwen2.5:32b","qwen2.5-coder:7b","qwen2.5-coder:14b","qwen2.5-coder:32b"]
!ollama pull {model}

]11;?\[GIN] 2026/11/26 - 22:46:26 | 200 |   2.69ms |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ 

time=2026-11-26T22:46:27.452Z level=INFO source=download.go:182 msg="downloading 96c415656d37 in 16 292 MB part(s)"


pulling manifest ⠙ pulling manifest 
pulling 96c415656d37:   0% ▕                  ▏ 7.1 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   1% ▕                  ▏  55 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   2% ▕                  ▏ 100 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   2% ▕                  ▏ 116 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   3% ▕                  ▏ 154 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   4% ▕                  ▏ 193 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   5% ▕                  ▏ 237 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   7% ▕█                 ▏ 312 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   7% ▕█                 ▏ 332 MB/4.7 GB                  pulling manifest 
pulling 96c415656d37:   8% ▕█                 ▏ 361 MB/4.7 GB                  pulling man

time=2026-11-26T22:48:03.197Z level=INFO source=download.go:182 msg="downloading c5ad996bda6e in 1 556 B part(s)"


pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         

time=2026-11-26T22:48:03.638Z level=INFO source=download.go:182 msg="downloading 6e4c38e1172f in 1 1.1 KB part(s)"


pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         

time=2026-11-26T22:48:04.179Z level=INFO source=download.go:182 msg="downloading f4d24e9138dd in 1 148 B part(s)"


pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         
pulling 40fb844194b2: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠋ pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         
pulling 40fb844194b2: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠙ pulling manifest 
pulling 96c415656d37: 100% ▕██████████████████▏ 4.7 GB                         
pulling 40fb844194b2: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest ⠹ 

In [15]:
! ollama show deepseek-r1:7b  #Technische Informationen zum heruntergeladenen KI-Modell anzeigen

]11;?\[GIN] 2026/11/26 - 22:53:19 | 200 |   1.45ms |       127.0.0.1 | HEAD     "/"
[GIN] 2026/11/26 - 22:53:19 | 200 | 187.93ms |       127.0.0.1 | POST     "/api/show"
  Model
    architecture        qwen2                 
    parameters          7.6B                  
    context length      131072                
    embedding length    3584                  
    quantization        Q4_K_M                
    template source     gguf_chat_template    

  Capabilities
    tools         
    thinking      
    completion    

  Parameters
    stop    "<｜begin▁of▁sentence｜>"    
    stop    "<｜end▁of▁sentence｜>"      
    stop    "<｜User｜>"                 
    stop    "<｜Assistant｜>"            

  License
    MIT License                    
    Copyright (c) 2023 DeepSeek    
    ...                            



In [ ]:
# @title Interacting with the model
question = "are you pretending to be a cat?"                                            # @param {"type":"string"}
from IPython.display import display, Markdown                                           #Funktionen zur formatierten Darstellung von Markdown in Jupyter importieren
import ollama                                                                           #Python-Bibliothek zur Kommunikation mit der Ollama API importieren
response = ollama.chat(model, messages=[                                                #Nachrichten an das ausgewählte lokale Modell senden und Antwortobjekt speichern
  { "role": "user", 'content:': "pretend to be a cat and add meows to all replies."},   #Anweisung zum gewünschten Verhalten des Modells
  {'role': 'user','content': question}                                                  #Eigentliche Benutzerfrage an das Modell übergeben
])
# print(response['message']['content'])
display(Markdown(response['message']['content']))                                       #Antworttext extrahieren und formatiert als Markdown darstellen

No, I don't have feelings or consciousness, so I can't pretend in any literal sense. However, if you're asking in a playful way, maybe we could explore some fun scenarios where an AI character pretends to be a cat! How about that?